# CLAUD·IA — Écouter le modèle avant et après le fine-tuning

Les deux versions reçoivent les **mêmes enregistrements personnels** et traduisent de l’espagnol vers l’anglais. Le modèle original et le modèle ajusté utilisent la même synthèse vocale. Les enregistrements personnels servent uniquement à cette évaluation.

[Documentation de SeamlessM4T v2](https://huggingface.co/docs/transformers/model_doc/seamless_m4t_v2)

## 1. Préparer Colab

Sélectionnez **T4 GPU** dans les paramètres d’exécution. Lancez l’installation. Après un éventuel redémarrage du noyau, importez `fine_tuning_pilote.zip` et les deux enregistrements dans le panneau **Fichiers**. L’archive doit rester compressée.

In [ ]:
%pip install "torch>=2.6" "transformers[sentencepiece]==5.22.1" "huggingface-hub>=1.32,<2"

## 2. Préparer les audios et lire les poids ajustés

FFmpeg récupère le son des fichiers MP4 ou WAV et le convertit en mono à 16 kHz en mémoire. `ZipFile` lit le rapport et les poids directement dans le ZIP exporté par le notebook 03.

Si un fichier porte un autre nom, modifiez la valeur dans `examples`.

In [ ]:
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile

import numpy as np
import torch
from safetensors.torch import load

if not torch.cuda.is_available():
    raise RuntimeError("Sélectionnez T4 GPU dans Colab.")
if shutil.which("ffmpeg") is None:
    raise RuntimeError("FFmpeg est nécessaire pour préparer les enregistrements.")

model_id = "facebook/seamless-m4t-v2-large"
sample_rate = 16_000
target_language = "eng"
device = torch.device("cuda")
archive_path = Path("fine_tuning_pilote.zip")
examples = [
    ("gracias_por_venir.mp4", "Gracias por venir.", "Thank you for coming."),
    ("antes_de_las_ocho.mp4", "No puedo llegar antes de las ocho.", "I can’t arrive before eight."),
]

def read_recording(filename):
    if not Path(filename).is_file():
        raise FileNotFoundError(f"Importez {filename} depuis le panneau Fichiers.")
    conversion = subprocess.run(
        ["ffmpeg", "-nostdin", "-v", "error", "-i", filename, "-vn", "-ac", "1",
         "-ar", str(sample_rate), "-f", "f32le", "pipe:1"],
        capture_output=True,
    )
    if conversion.returncode != 0:
        raise RuntimeError(conversion.stderr.decode("utf-8", errors="replace"))
    audio = np.frombuffer(conversion.stdout, dtype="<f4").copy()
    if audio.size == 0:
        raise ValueError(f"Aucun son dans {filename}.")
    return audio

if not archive_path.is_file():
    raise FileNotFoundError("Importez fine_tuning_pilote.zip depuis le panneau Fichiers.")

with ZipFile(archive_path) as archive:
    report = json.loads(archive.read("rapport.json"))
    adjusted_weights = load(archive.read("poids_ajustes.safetensors"))

if (report["base_model"], report["source_language"], report["target_language"]) != (
    model_id, "spa", target_language
):
    raise ValueError("L’archive doit correspondre au fine-tuning espagnol → anglais du notebook 03.")

recordings = [(source, reference, read_recording(filename)) for filename, source, reference in examples]
print(torch.cuda.get_device_name(0))
print(f"{len(recordings)} enregistrements et {len(adjusted_weights)} tenseurs chargés.")

## 3. Charger le traducteur vocal

Nous chargeons le modèle vocal une seule fois et gardons une copie de ses quatre tenseurs d’origine. Les poids ajustés ont été sauvegardés en FP32. Nous les convertissons en FP16, comme les autres poids du traducteur sur T4. Le notebook vérifie que des valeurs diffèrent encore après la conversion.

In [ ]:
from transformers import AutoProcessor, SeamlessM4Tv2ForSpeechToSpeech

processor = AutoProcessor.from_pretrained(model_id)
model = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
    model_id, dtype=torch.float16, use_safetensors=True
).to(device)
model.eval()

layer_index = len(model.text_decoder.layers) - 1
prefix = f"text_decoder.layers.{layer_index}.cross_attention."
expected_names = {
    f"{prefix}{projection}.{kind}"
    for projection in ("q_proj", "v_proj")
    for kind in ("weight", "bias")
}
if adjusted_weights.keys() != expected_names:
    raise ValueError("Les noms des poids ne correspondent pas aux quatre tenseurs du pilote.")

parameters = {name: model.get_parameter(name) for name in expected_names}
original_weights = {name: parameter.detach().cpu().clone() for name, parameter in parameters.items()}
for name, weights in adjusted_weights.items():
    if weights.shape != parameters[name].shape or not torch.isfinite(weights).all():
        raise ValueError(f"Tenseur incompatible : {name}")

adjusted_weights = {
    name: weights.to(dtype=parameters[name].dtype) for name, weights in adjusted_weights.items()
}
if all(torch.equal(original_weights[name], weights) for name, weights in adjusted_weights.items()):
    raise ValueError("Les poids ajustés sont identiques aux poids d’origine en FP16.")

@torch.no_grad()
def set_weights(weights):
    for name, parameter in parameters.items():
        parameter.copy_(weights[name])

print("Modèle vocal chargé. Les poids ajustés correspondent au décodeur.")

## 4. Comparer les traductions

Pour chaque phrase, écoutez l’original puis les deux traductions. Vérifiez le sens, la négation et les nombres. Deux formulations différentes peuvent être correctes. Ces deux phrases servent de premier test sur nos enregistrements.

In [ ]:
from IPython.display import Audio, display

@torch.inference_mode()
def translate(audio):
    inputs = processor(audio=audio, sampling_rate=sample_rate, return_tensors="pt").to(device)
    inputs["input_features"] = inputs["input_features"].to(dtype=torch.float16)
    result = model.generate(
        **inputs, tgt_lang=target_language, return_intermediate_token_ids=True,
        text_do_sample=False, speech_do_sample=False,
    )
    text = processor.batch_decode(
        result.sequences.cpu(), skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]
    length = result.waveform_lengths.item()
    waveform = result.waveform.reshape(-1)[:length].float().cpu().numpy()
    return text, waveform

set_weights(original_weights)
before = [translate(audio) for _, _, audio in recordings]
try:
    set_weights(adjusted_weights)
    after = [translate(audio) for _, _, audio in recordings]
finally:
    set_weights(original_weights)

for (source, reference, audio), original, tuned in zip(recordings, before, after):
    print(f"Phrase : {source}")
    print(f"Référence : {reference}")
    print("Audio original")
    display(Audio(audio, rate=sample_rate))
    for label, (text, waveform) in (("Avant", original), ("Après", tuned)):
        print(f"{label} : {text}")
        display(Audio(waveform, rate=model.config.sampling_rate))

## 5. Comparer une voix normale et une voix basse

Dans une session qui a déjà terminé la section 4, copiez seulement la cellule suivante sous les résultats. Elle réutilise le modèle chargé.

Enregistrez deux fois **la même phrase**, avec le même micro, à la même distance et dans le même endroit. Parlez normalement dans le premier fichier, puis plus bas dans le second. Importez les deux MP4. Le niveau RMS indiqué décrit le signal enregistré. Il permet de vérifier si le fichier « voix basse » a effectivement un niveau plus faible.

| Phrase à enregistrer | Traduction anglaise de référence |
| --- | --- |
| No puedo llegar antes de las ocho. | I can’t arrive before eight. |
| Gracias por venir. | Thank you for coming. |
| Necesito dos boletos para mañana. | I need two tickets for tomorrow. |
| El tren sale a las siete y media. | The train leaves at half past seven. |
| La farmacia está al lado del banco. | The pharmacy is next to the bank. |
| No olvides llevar el pasaporte. | Don’t forget to bring your passport. |

Pour chaque prise, écoutez les versions avant et après fine-tuning. Attribuez **2** si le sens est conservé, **1** si une information manque et **0** si le sens change ou si l’audio est inutilisable. Une formulation anglaise différente peut être correcte. Répétez la cellule avec les noms de fichiers et la référence de chaque phrase. Notez les scores par condition et par version.

In [ ]:
reference = "I can’t arrive before eight."
voice_files = [
    ("Voix normale", "ocho_normal.mp4"),
    ("Voix basse", "ocho_baja.mp4"),
]
voice_recordings = [(label, read_recording(filename)) for label, filename in voice_files]

set_weights(original_weights)
voice_before = [translate(audio) for _, audio in voice_recordings]
try:
    set_weights(adjusted_weights)
    voice_after = [translate(audio) for _, audio in voice_recordings]
finally:
    set_weights(original_weights)

print(f"Référence : {reference}")
for (condition, audio), before_result, after_result in zip(voice_recordings, voice_before, voice_after):
    rms = np.sqrt(np.mean(audio.astype(np.float64) ** 2))
    rms_dbfs = 20 * np.log10(max(rms, 1e-12))
    print(f"\n{condition} • niveau RMS : {rms_dbfs:.1f} dBFS")
    for label, (text, waveform) in (("Avant", before_result), ("Après", after_result)):
        print(f"{label} : {text}")
        display(Audio(waveform, rate=model.config.sampling_rate))